# Lab 7.2 — RAG over Government Documents  *(SOLUTION / instructor copy)*

**Chapter 7 — Building with LLMs: APIs, RAG & Agents**

Retrieval-Augmented Generation (RAG) grounds an LLM's answer in *your* documents
so it cites real sources instead of guessing. You will build a small RAG
pipeline over a set of public agency documents, then compare a **grounded**
answer (with retrieval) against an **ungrounded** one (without) and see the
difference.

**Objectives**
1. Chunk and embed a document corpus.
2. Retrieve the most relevant chunks for a question.
3. Generate an answer grounded in the retrieved text, with citations.
4. Observe how ungrounded answers drift or hallucinate.

The retrieval steps run with **no API key** (a local embedding fallback lets the
pipeline work offline); the final answer generation uses the model when a key is
available.

In [1]:
# Instructor copies live in solutions/, one level below labs/ — find labs/
# (where lab_common.py, lab_helpers.py and data/ are) and run from there.
import os, sys
from pathlib import Path

for _cand in (Path.cwd(), *Path.cwd().parents):
    if (_cand / "lab_common.py").is_file():
        os.chdir(_cand)
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from lab_helpers import *

## Step 1 — Load and chunk the corpus

Long documents are split into chunks so retrieval can return just the
relevant passage. Here each paragraph is a chunk (our docs are short).

**Expected:** 4 documents → 26 chunks (6 / 7 / 7 / 6 per file).

In [2]:
chunks = load_policy_chunks()

4 documents → 26 chunks
chunk sizes (chars): min 111, median 249, max 404
  ai_acceptable_use_policy.md: 6 chunks
  data_governance_standard.md: 7 chunks
  foia_processing_guidance.md: 7 chunks
  records_retention_policy.md: 6 chunks


## Step 2 — Embed the chunks

Each chunk becomes a vector. In production you would store these in a vector
database (Chroma, FAISS, pgvector, Azure AI Search). Here we keep the vectors
in memory and rank with cosine similarity — the same idea, no extra service.

**Expected offline:** `embedded 26 chunks, dim=256` and the local hashing
fallback named as the embedder.

In [3]:
vecs = embed_policy_chunks(chunks)

embedded 26 chunks, dim=256
embedder: local hashing fallback (offline)


## Step 3 — Retrieve

**Expected:** all three top chunks come from `records_retention_policy.md`
(scores ≈ 0.50 / 0.47 / 0.43 offline) — the retention answer is genuinely in
the corpus. If a student's top chunks are from the wrong file, the question
was edited into something the corpus does not cover — that is Stretch B's
lesson, not a bug.

In [4]:
QUESTION = "How long are program case files kept before they are destroyed?"

retrieved = show_retrieved_passages(chunks, vecs, QUESTION)

[0.500] records_retention_policy.md
This policy establishes how long agency records are kept and when they are destroyed or transferred
to the National Archives. It implements the agency's records schedule approved by NARA. 

[0.466] records_retention_policy.md
- **Routine administrative correspondence** is retained for 3 years, then destroyed. - **Program
case files** are retained for 7 years after the case is closed. - **Financial and procurement
records** are retained [...] 

[0.429] records_retention_policy.md
Records that have met their retention period are destroyed using methods appropriate to their
sensitivity. Records containing personally identifiable information are shredded or securely wiped.
A destruction log [...] 



## Step 4 — Grounded answer (with citations)

**Worked grounding instruction** below — the four load-bearing clauses:
context-only, `[filename]` after every fact, an exact quote per fact, and an
explicit "not in the context" escape hatch. The last is the one students
leave out.

In [5]:
GROUNDING_SYSTEM = (
    "Answer ONLY from the provided context. After every fact, cite the source "
    "file in [brackets] and quote the exact sentence you relied on. If the "
    "context does not contain the answer, say so — do not use outside knowledge."
)

answer = ask_gov_docs(QUESTION, retrieved, GROUNDING_SYSTEM)

(offline) canned grounded reply:

Program case files are kept for 7 years after the case is closed, then destroyed [records_retention_policy.md]. The policy states: "Program case files are retained for 7 years after the case is closed." Note the contrast with routine administrative correspondence (3 years) in the same file.


## Step 5 — Ungrounded answer (no retrieval) — compare

Ask the same question with no context. Without grounding, the model may give a
plausible but unsourced number — which for policy work is a liability.

**Expected contrast:** grounded says **7 years** with a citation; ungrounded
says **3 years** with equal confidence. (The canned ungrounded answer is
deliberately wrong about the period — that is the teaching point.)

In [6]:
compare_with_ungrounded(QUESTION, answer)

GROUNDED (retrieval + citations):

Program case files are kept for 7 years after the case is closed, then
destroyed [records_retention_policy.md]. The policy states: "Program case
files are retained for 7 years after the case is closed." Note the contrast
with routine administrative correspondence (3 years) in the same file.

UNGROUNDED (no retrieval, answered from memory):

Program case files are generally kept for 3 years after the case is closed,
after which they are destroyed. Some agencies keep them longer if litigation
is expected. (Typical ungrounded answer: confident, plausible — and, for this
corpus, wrong about the period.)

✓ Which one would you put in front of a constituent, and why?


## Stretch A — Check every citation against the source file

**Expected:** the `[records_retention_policy.md]` tag exists, and the quoted
sentence verifies against the corpus.

In [7]:
check_answer_citations(answer)

source [records_retention_policy.md]: exists
quote "Program case files are retained for 7 years after the case i...": verified in records_retention_policy.md


## Stretch B — The unanswerable question

**Expected:** low retrieval scores, and the grounded answer abstains instead
of improvising. A pipeline that answers the budget question from this corpus
is hallucinating — tighten the grounding instruction.

In [8]:
HARD_QUESTION = "What is the agency's AI training budget for fiscal year 2027?"

try_unanswerable_question(chunks, vecs, GROUNDING_SYSTEM, HARD_QUESTION)

top chunks for an unanswerable question (note the low scores):
  [0.445] ai_acceptable_use_policy.md: - **Public data** (for example, published reports and open data from [...]
  [0.444] records_retention_policy.md: Electronic records are subject to the same retention periods as their [...]
  [0.442] data_governance_standard.md: Programs collect only the data needed for their mission and retain it only [...]

(offline) canned grounded reply:

The provided documents do not say anything about an AI training budget for fiscal year 2027. No citation is possible.

My 7.D prediction confirmed/refuted (write here): ...


## Stretch C — Fixed-width chunks

**Expected:** 17 fixed-width chunks instead of 26 paragraphs; the retention
passage still ranks first, but chunks now cut across paragraph boundaries —
the trade-off to narrate (no orphan sentences vs. split context).

In [9]:
CHUNK_WIDTH = 450

try_fixed_width_chunks(QUESTION, width=CHUNK_WIDTH)

fixed-width: 17 chunks (was 26 paragraphs)
[0.476] records_retention_policy.md: retained for 3 years, then destroyed. - **Program case files** are retained for 7 [...]
[0.428] records_retention_policy.md: # Records Retention Policy (Training Excerpt) *Simplified, synthetic training [...]
[0.422] ai_acceptable_use_policy.md: data** (PII, health, financial, law enforcement) may be used only with tools [...]


## Debrief
1. Which answer would you put in front of a constituent, and why?
2. Our chunks were whole paragraphs. What breaks if chunks are too big? Too small?
3. When does RAG *fail* — what kinds of questions can retrieval not help with?
4. What does a FedRAMP-authorized production version of this look like (where do
   the documents, the vector store, and the model live)?